# Urban Infrastructure Pulse Intelligence

## NYC 311 Service Requests — Data Extraction

### Project Objective
The objective of this stage is to collect a representative sample of New York City 311 service requests for analyzing urban service patterns, identifying potential infrastructure concerns, and supporting subsequent unsupervised machine learning.

### Data Source
- **Source:** NYC Open Data — 311 Service Requests
- **Dataset ID:** `erm2-nwe9`
- **Study period:** January 2023 – December 2025
- **Target sample size:** 60,000 records

### Extraction Strategy
The dataset is accessed through the NYC Open Data API. To avoid downloading millions of records, the extraction uses stratification by month and borough, with sample allocations based on the corresponding request populations.

### Selected Attributes
Request ID, creation date, closure date, agency, complaint type, descriptor, borough, incident ZIP code, latitude, and longitude.

### Expected Output
A CSV dataset containing 60,000 unique service requests for subsequent data cleaning, exploratory data analysis, feature engineering, and unsupervised machine learning.

## 1. API Connection Test

In [1]:
import requests
import pandas as pd

# Official NYC 311 dataset API
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

params = {
    "$select": (
        "unique_key,created_date,closed_date,"
        "agency,complaint_type,descriptor,"
        "borough,incident_zip,latitude,longitude"
    ),
    "$where": (
        "created_date >= '2023-01-01T00:00:00' "
        "AND created_date < '2026-01-01T00:00:00'"
    ),
    "$order": "created_date",
    "$limit": 5
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

df_test = pd.DataFrame(response.json())

print("Rows retrieved:", len(df_test))
print("Columns:", df_test.columns.tolist())
display(df_test.head())
print(df_test.dtypes)


Rows retrieved: 5
Columns: ['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'borough', 'incident_zip', 'latitude', 'longitude']


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,latitude,longitude
0,56416396,2023-01-01T00:00:00.000,2023-01-03T14:56:48.000,DOHMH,Food Poisoning,1 or 2,QUEENS,11379,40.72762987830367,-73.87361356231048
1,56417527,2023-01-01T00:00:09.000,2023-01-01T00:36:06.000,NYPD,Illegal Fireworks,N/A,BROOKLYN,11218,40.640914779776715,-73.97364216306418
2,56416252,2023-01-01T00:00:42.000,2023-01-01T17:34:15.000,NYPD,Noise - Residential,Loud Music/Party,BRONX,10453,40.85384789145288,-73.91770920479841
3,56418795,2023-01-01T00:00:45.000,2023-01-01T01:24:10.000,NYPD,Illegal Parking,Posted Parking Sign Violation,MANHATTAN,10001,40.75487501846257,-74.00074715055744
4,56418136,2023-01-01T00:00:46.000,2023-01-01T01:01:43.000,NYPD,Noise - Residential,Loud Music/Party,BROOKLYN,11234,40.62066491675458,-73.92604033064251


unique_key        object
created_date      object
closed_date       object
agency            object
complaint_type    object
descriptor        object
borough           object
incident_zip      object
latitude          object
longitude         object
dtype: object


## 2. Population Analysis and Sample Allocation

In [2]:
import requests
import pandas as pd
import math

URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
TARGET_SIZE = 60000

params = {
    "$select": (
        "date_extract_y(created_date) as year, "
        "date_extract_m(created_date) as month, "
        "borough, count(*) as population"
    ),
    "$where": (
        "created_date >= '2023-01-01T00:00:00' "
        "AND created_date < '2026-01-01T00:00:00'"
    ),
    "$group": (
        "date_extract_y(created_date), "
        "date_extract_m(created_date), borough"
    ),
    "$order": "year, month, borough",
    "$limit": 1000
}

response = requests.get(URL, params=params, timeout=120)
response.raise_for_status()

strata_df = pd.DataFrame(response.json())

strata_df["year"] = strata_df["year"].astype(int)
strata_df["month"] = strata_df["month"].astype(int)
strata_df["population"] = pd.to_numeric(
    strata_df["population"]
)

population_total = int(strata_df["population"].sum())

# Proportional allocation
strata_df["raw_allocation"] = (
    strata_df["population"] / population_total * TARGET_SIZE
)

strata_df["sample_size"] = (
    strata_df["raw_allocation"].apply(math.floor)
)

# Largest-remainder method ensures exactly 60,000
remaining = TARGET_SIZE - int(strata_df["sample_size"].sum())

strata_df["fraction"] = (
    strata_df["raw_allocation"] - strata_df["sample_size"]
)

indices = strata_df["fraction"].sort_values(
    ascending=False
).index[:remaining]

strata_df.loc[indices, "sample_size"] += 1

strata_df["period"] = (
    strata_df["year"].astype(str)
    + "-"
    + strata_df["month"].astype(str).str.zfill(2)
)

print("Total population:", population_total)
print("Number of strata:", len(strata_df))
print("Total allocated sample:", strata_df["sample_size"].sum())

print("\nSample allocation by month:")
display(
    strata_df.groupby("period")["sample_size"]
    .sum()
    .reset_index()
)

print("\nSample allocation by borough:")
display(
    strata_df.groupby("borough", dropna=False)["sample_size"]
    .sum()
    .reset_index()
)

print("\nFirst 15 strata:")
display(
    strata_df[
        ["year", "month", "borough", "population", "sample_size"]
    ].head(15)
)

# Final allocation checks
assert len(strata_df) > 0, "No population data returned."
assert not strata_df.duplicated(
    subset=["year", "month", "borough"]
).any(), "Duplicate strata found."
assert strata_df["population"].gt(0).all(), "Invalid population count."
assert strata_df["sample_size"].sum() == TARGET_SIZE, (
    "Sample allocation does not equal 60,000."
)

print("\nAll population and allocation checks passed.")

Total population: 10336535
Number of strata: 216
Total allocated sample: 60000

Sample allocation by month:


,period,sample_size
0,2023-01,1407
1,2023-02,1305
2,2023-03,1461
3,2023-04,1437
4,2023-05,1632
5,2023-06,1549
6,2023-07,1679
7,2023-08,1640
8,2023-09,1646
9,2023-10,1758



Sample allocation by borough:


,borough,sample_size
0,BRONX,12782
1,BROOKLYN,18136
2,MANHATTAN,12384
3,QUEENS,14409
4,STATEN ISLAND,2229
5,Unspecified,60



First 15 strata:


,year,month,borough,population,sample_size
0,2023,1,BRONX,51779,301
1,2023,1,BROOKLYN,74523,433
2,2023,1,MANHATTAN,48822,283
3,2023,1,QUEENS,55779,324
4,2023,1,STATEN ISLAND,10637,62
5,2023,1,Unspecified,633,4
6,2023,2,BRONX,49284,286
7,2023,2,BROOKLYN,69210,402
8,2023,2,MANHATTAN,46596,270
9,2023,2,QUEENS,50657,294



All population and allocation checks passed.


## 3. Stratified Data Extraction

**Note:** This cell contains the code used to extract the stratified sample from the NYC Open Data API. Running it again will repeat the extraction process and may overwrite the existing CSV file. The validated dataset is already saved as `nyc_311_stratified_sample_v2.csv`, so this cell does not need to be rerun for subsequent analysis.

In [6]:
import requests
import pandas as pd
import random
import math
import time
import os

URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

BATCH_SIZE = 500
CHECKPOINT_DIR = "nyc311_checkpoints"
OUTPUT_FILE = "nyc_311_stratified_sample_v2.csv"
SEED = 42

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

rng = random.Random(SEED)
session = requests.Session()

columns = [
    "unique_key", "created_date", "closed_date",
    "agency", "complaint_type", "descriptor",
    "borough", "incident_zip", "latitude", "longitude"
]


def fetch_records(params, retries=5):
    for attempt in range(retries):
        try:
            response = session.get(URL, params=params, timeout=120)

            if response.ok:
                return response.json()

            if response.status_code not in [429, 500, 502, 503, 504]:
                response.raise_for_status()

        except requests.RequestException:
            if attempt == retries - 1:
                raise

        wait = min(2 ** attempt, 30)
        print(f"  Request failed; retrying in {wait}s...")
        time.sleep(wait)

    raise RuntimeError("API request failed after retries.")


def get_stratum_filter(year, month, borough):
    start = f"{year}-{month:02d}-01T00:00:00"

    if month == 12:
        end = f"{year + 1}-01-01T00:00:00"
    else:
        end = f"{year}-{month + 1:02d}-01T00:00:00"

    if borough == "Unspecified":
        borough_condition = (
            "borough IS NULL OR borough = '' "
            "OR borough = 'Unspecified'"
        )
    else:
        borough_condition = f"borough = '{borough}'"

    return (
        f"created_date >= '{start}' "
        f"AND created_date < '{end}' "
        f"AND ({borough_condition})"
    )


selected_groups = []

for i, row in strata_df.iterrows():
    year = int(row["year"])
    month = int(row["month"])
    borough = row["borough"]
    population = int(row["population"])
    required = int(row["sample_size"])

    if required == 0:
        continue

    checkpoint_file = os.path.join(
        CHECKPOINT_DIR,
        f"{year}_{month:02d}_{i}.csv"
    )

    print(f"\n{year}-{month:02d} | {borough} | target={required}")

    # Reuse a completed stratum if it was saved previously.
    if os.path.exists(checkpoint_file):
        saved = pd.read_csv(checkpoint_file)

        if (
            len(saved) == required
            and saved["unique_key"].nunique() == required
        ):
            selected_groups.append(saved)
            print("  Loaded checkpoint:", len(saved))
            continue
        else:
            print("  Invalid checkpoint; extracting this stratum again.")
            os.remove(checkpoint_file)

    where_clause = get_stratum_filter(year, month, borough)

    # Get the current number of records in this stratum.
    count_result = fetch_records({
        "$select": "count(*) as n",
        "$where": where_clause
    })
    current_population = int(count_result[0]["n"])

    if current_population < required:
        raise RuntimeError(
            f"Only {current_population} records available for "
            f"{year}-{month:02d}, {borough}; need {required}."
        )

    # Choose random offsets and avoid fetching the same block twice.
    total_blocks = math.ceil(current_population / BATCH_SIZE)
    block_order = rng.sample(range(total_blocks), total_blocks)
    candidates = []
    seen_ids = set()

    for block_id in block_order:
        records = fetch_records({
            "$select": ",".join(columns),
            "$where": where_clause,
            "$order": "created_date,unique_key",
            "$limit": BATCH_SIZE,
            "$offset": block_id * BATCH_SIZE
        })

        for record in records:
            record_id = record.get("unique_key")
            if record_id is not None and record_id not in seen_ids:
                candidates.append(record)
                seen_ids.add(record_id)

        if len(candidates) >= required:
            break

        time.sleep(0.1)

    candidate_df = pd.DataFrame(candidates)

    if len(candidate_df) < required:
        raise RuntimeError(
            f"Not enough unique candidates for "
            f"{year}-{month:02d}, {borough}."
        )

    sampled = candidate_df.sample(
        n=required,
        random_state=SEED + i
    ).copy()

    # Save immediately, before proceeding to the next stratum.
    sampled.to_csv(checkpoint_file, index=False)
    selected_groups.append(sampled)

    print("  Candidates:", len(candidate_df))
    print("  Saved sample:", len(sampled))


# Combine all strata only after extraction succeeds.
final_df_v2 = pd.concat(selected_groups, ignore_index=True)

final_df_v2 = final_df_v2.drop_duplicates(subset="unique_key")

if len(final_df_v2) != TARGET_SIZE:
    raise RuntimeError(
        f"Expected {TARGET_SIZE} rows; got {len(final_df_v2)}."
    )

final_df_v2["created_date"] = pd.to_datetime(
    final_df_v2["created_date"], errors="coerce"
)
final_df_v2["closed_date"] = pd.to_datetime(
    final_df_v2["closed_date"], errors="coerce"
)

for col in ["latitude", "longitude"]:
    final_df_v2[col] = pd.to_numeric(
        final_df_v2[col], errors="coerce"
    )

final_df_v2 = final_df_v2.sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

final_df_v2.to_csv(OUTPUT_FILE, index=False)

print("\nEXTRACTION FINISHED")
print("Rows:", len(final_df_v2))
print("Unique IDs:", final_df_v2["unique_key"].nunique())
print("Checkpoint folder:", CHECKPOINT_DIR)
print("Output file:", OUTPUT_FILE)

display(final_df_v2.head())



2023-01 | BRONX | target=301
  Loaded checkpoint: 301

2023-01 | BROOKLYN | target=433
  Loaded checkpoint: 433

2023-01 | MANHATTAN | target=283
  Loaded checkpoint: 283

2023-01 | QUEENS | target=324
  Loaded checkpoint: 324

2023-01 | STATEN ISLAND | target=62
  Loaded checkpoint: 62

2023-01 | Unspecified | target=4
  Candidates: 500
  Saved sample: 4

2023-02 | BRONX | target=286
  Candidates: 500
  Saved sample: 286

2023-02 | BROOKLYN | target=402
  Candidates: 500
  Saved sample: 402

2023-02 | MANHATTAN | target=270
  Candidates: 500
  Saved sample: 270

2023-02 | QUEENS | target=294
  Candidates: 500
  Saved sample: 294

2023-02 | STATEN ISLAND | target=52
  Candidates: 500
  Saved sample: 52

2023-02 | Unspecified | target=1
  Candidates: 253
  Saved sample: 1

2023-03 | BRONX | target=289
  Candidates: 500
  Saved sample: 289

2023-03 | BROOKLYN | target=457
  Candidates: 500
  Saved sample: 457

2023-03 | MANHATTAN | target=307
  Candidates: 500
  Saved sample: 307

2023-

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,latitude,longitude
0,58747927,2023-09-07 21:18:18,2024-01-19 07:29:19,EDC,Noise - Helicopter,Other,BROOKLYN,11238,40.672162,-73.964289
1,63535037,2024-12-27 15:50:00,2024-12-29 01:10:00,DEP,Water System,Hydrant Leaking (WC1),BROOKLYN,11211,40.709141,-73.954578
2,63889795,2025-01-25 20:31:17,2025-01-25 21:46:14,NYPD,Noise - Residential,Banging/Pounding,BROOKLYN,11216,40.691099,-73.947306
3,57928606,2023-06-18 01:19:59,2023-06-18 01:30:47,NYPD,Noise - Residential,Loud Music/Party,QUEENS,11373,40.733078,-73.880744
4,57899699,2023-06-14 18:14:07,2023-06-14 18:31:16,NYPD,Illegal Parking,Posted Parking Sign Violation,MANHATTAN,10033,40.846116,-73.940204


In [7]:

import pandas as pd
import os

file_path = "nyc_311_stratified_sample_v2.csv"

df = pd.read_csv(file_path)

print("File exists:", os.path.exists(file_path))
print("Dataset shape:", df.shape)
print("Unique complaint IDs:", df["unique_key"].nunique())

print("\nColumn names:")
print(df.columns.tolist())

print("\nDate range:")
print("Start:", df["created_date"].min())
print("End:", df["created_date"].max())

print("\nComplaints by borough:")
print(df["borough"].value_counts(dropna=False))

print("\nMissing values:")
print(df.isnull().sum())

print("\nFirst 5 rows:")
display(df.head())


File exists: True
Dataset shape: (60000, 10)
Unique complaint IDs: 60000

Column names:
['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'borough', 'incident_zip', 'latitude', 'longitude']

Date range:
Start: 2023-01-02 15:06:52
End: 2025-12-30 18:19:42

Complaints by borough:
borough
BROOKLYN         18136
QUEENS           14409
BRONX            12782
MANHATTAN        12384
STATEN ISLAND     2229
Unspecified         60
Name: count, dtype: int64

Missing values:
unique_key           0
created_date         0
closed_date       1078
agency               0
complaint_type       0
descriptor        1939
borough              0
incident_zip       601
latitude           935
longitude          935
dtype: int64

First 5 rows:


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,latitude,longitude
0,58747927,2023-09-07 21:18:18,2024-01-19 07:29:19,EDC,Noise - Helicopter,Other,BROOKLYN,11238.0,40.672162,-73.964289
1,63535037,2024-12-27 15:50:00,2024-12-29 01:10:00,DEP,Water System,Hydrant Leaking (WC1),BROOKLYN,11211.0,40.709141,-73.954578
2,63889795,2025-01-25 20:31:17,2025-01-25 21:46:14,NYPD,Noise - Residential,Banging/Pounding,BROOKLYN,11216.0,40.691099,-73.947306
3,57928606,2023-06-18 01:19:59,2023-06-18 01:30:47,NYPD,Noise - Residential,Loud Music/Party,QUEENS,11373.0,40.733078,-73.880744
4,57899699,2023-06-14 18:14:07,2023-06-14 18:31:16,NYPD,Illegal Parking,Posted Parking Sign Violation,MANHATTAN,10033.0,40.846116,-73.940204


## 4. Final Dataset Validation

The extracted dataset is loaded from the saved CSV file and checked for record count, duplicate request IDs, date coverage, missing values, and distribution across boroughs and years.

This step verifies the saved dataset without making additional API requests.

In [8]:
import pandas as pd

# Load the previously extracted dataset
df = pd.read_csv("nyc_311_stratified_sample_v2.csv")

# Convert date columns
df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
df["closed_date"] = pd.to_datetime(df["closed_date"], errors="coerce")

# Basic validation
print("Total records:", len(df))
print("Unique request IDs:", df["unique_key"].nunique())
print("Duplicate request IDs:", df["unique_key"].duplicated().sum())
print("Date range:", df["created_date"].min(), "to", df["created_date"].max())

print("\nRecords by borough:")
print(df["borough"].value_counts(dropna=False))

print("\nRecords by year:")
print(df["created_date"].dt.year.value_counts().sort_index())

print("\nMissing values:")
print(df.isnull().sum())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Total records: 60000
Unique request IDs: 60000
Duplicate request IDs: 0
Date range: 2023-01-02 15:06:52 to 2025-12-30 18:19:42

Records by borough:
borough
BROOKLYN         18136
QUEENS           14409
BRONX            12782
MANHATTAN        12384
STATEN ISLAND     2229
Unspecified         60
Name: count, dtype: int64

Records by year:
created_date
2023    18718
2024    20067
2025    21215
Name: count, dtype: int64

Missing values:
unique_key           0
created_date         0
closed_date       1078
agency               0
complaint_type       0
descriptor        1939
borough              0
incident_zip       601
latitude           935
longitude          935
dtype: int64
Dataset shape: (60000, 10)
Columns: ['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'borough', 'incident_zip', 'latitude', 'longitude']


### Validation Summary

The final dataset contains 60,000 NYC 311 service requests collected from January 2023 through December 2025. All request IDs are unique, and records cover all five NYC boroughs, along with a small number of requests with unspecified boroughs.

Missing values occur in closure dates, descriptors, ZIP codes, and geographic coordinates. These will be assessed and handled during the data-cleaning and feature-engineering stage.

The dataset is ready for exploratory data analysis, subject to further checks of sampling quality and data suitability for the project's analytical objectives.""